In [37]:
import pandas as pd
import random
from pycaret.classification import *
import numpy as np

In [1]:
def add_class_col(df):
    labels = []
    for region in df["non_rep_region"]:
        if region[-3:] == "pos":
            labels.append(1)
        elif region[-3:] == "neg":
            labels.append(0)
    df["Class"] = labels

In [59]:
def setup_data(type):
    validate_data = pd.read_csv(f"input_data\\validate_chain_{type}_50-50_input_ready.csv")
    train_data = pd.read_csv(f"input_data\\train_chain_{type}_50-50_input_ready.csv")
    validate_data = validate_data.drop("non_rep_region", axis=1)
    train_data = train_data.drop("non_rep_region", axis=1)

    data = pd.concat([validate_data, train_data], ignore_index=True)

    s = setup(data=data, target="acr_label", test_data=validate_data, index=False, verbose=False)
    return s

In [ ]:
setup_data("rand")
best = compare_models()

In [ ]:
setup_data("rand")
model = create_model("gbc", verbose=False)
tuned = tune_model(model, optimize="F1")
tuned

In [ ]:
s = setup_data("up")
features = s._fxs['Numeric']
models = ["lightgbm", "gbc", "ada", "rf", "et"]
frequency = {}
for i in range(5):
    for model in models:
        model_ob = create_model(model, verbose=False)
        try:
            importance = model_ob.feature_importances_
        except:
            print(f"{model} doesn't have feature importance")
            continue
        importance_args = np.argsort(importance)[:5]
        for indx in importance_args:
            frequency[indx] = frequency.get(indx, 0) + 1
    print(f"Iteration {i} complete")
for indx, freq in frequency.items():
    print(f"{features[indx]} {freq / 5}")
print(frequency)

